In [295]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import math
from sklearn.model_selection import train_test_split
from functools import reduce
from functools import partial
import operator
from timeit import default_timer
from matplotlib.ticker import FormatStrFormatter
import deepxde as dde
import scipy.io as scio

In [296]:
# Parameters
epochs =1000
batch_size = 64
gamma = 0.5
learning_rate = 0.001
step_size= 50
modes=12
width=32
dim_x = 1
activation = "relu"
kernel = "Glorot normal"
learning_rate=0.001

In [297]:
# Create train/test splits
dataFile = "zx0"  
data = scio.loadmat(dataFile)
inpArr = data['zx0']
dataFile = "sigmax0"  
data = scio.loadmat(dataFile)
outArr = data['sigmax0']
x = np.array(inpArr, dtype=np.float32)
y = np.array(outArr, dtype=np.float32)

x = x.reshape(-1,1)
y = y.reshape(-1,1)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.1, random_state=1)
x_train = torch.from_numpy(x_train).cuda()
y_train = torch.from_numpy(y_train).cuda()
x_test = torch.from_numpy(x_test).cuda()
y_test = torch.from_numpy(y_test).cuda()

trainData = DataLoader(TensorDataset(x_train, y_train), batch_size=batch_size, shuffle=True, generator=torch.Generator(device='cuda'))
testData = DataLoader(TensorDataset(x_test, y_test), batch_size=batch_size, shuffle=False, generator=torch.Generator(device='cuda'))

In [298]:
def count_params(model):
    pp=0
    for p in list(model.parameters()):
        nn=1
        for s in list(p.size()):
            nn = nn*s
        pp += nn
    return pp

In [302]:
class mymodel(nn.Module):
    def __init__(self, activation, kernel):
        super(mymodel, self).__init__()
        self.fc1 = nn.Linear(1, 20)
        self.fc2 = nn.Linear(20, 20)
        self.fc3 = nn.Linear(20, 20)
        self.fc4 = nn.Linear(20, 20)
        self.fc5 = nn.Linear(20, 1)
        self.tanh = torch.tanh
        
    def forward(self, x):
        x = self.tanh(self.fc1(x))
        x = self.tanh(self.fc2(x))
        x = self.tanh(self.fc3(x))
        x = self.tanh(self.fc4(x))
        x = self.fc5(x)
        return x
       
model = mymodel(activation, kernel)
print(count_params(model))

1321


In [303]:
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)

In [304]:
loss = nn.MSELoss()
train_lossArr = []
test_lossArr = []
time_Arr = []

for ep in range(epochs):
    model.train()
    t1 = default_timer()
    train_loss = 0
    for x, y in trainData:
        x, y = x.cuda(), y.cuda()
        optimizer.zero_grad()
        out = model(x)
        lp = loss(out, y)
        lp.backward()
        
        optimizer.step()
        train_loss += lp.item()
        
    scheduler.step()
    model.eval()
    test_loss = 0
    with torch.no_grad():
        for x, y in testData:
            x, y = x.cuda(), y.cuda()
            out = model(x)
            test_loss += loss(out, y).item()
            
    train_loss /= len(trainData)
    test_loss /= len(testData)
    
    train_lossArr.append(train_loss)
    test_lossArr.append(test_loss)
    
    t2 = default_timer()
    time_Arr.append(t2-t1)
    if ep%50 == 0:
        print(ep, t2-t1, np.mean(train_lossArr[-50:]), np.mean(test_lossArr[-50:]))

0 0.08219629999803146 2.6161662936210632 2.5096306800842285
50 0.02012749999994412 0.39504061244428157 0.36790867686271667
100 0.031108499999390915 0.0810649969917722 0.06927462862804531
150 0.02035599999362603 0.00489003028604202 0.007264701910316944
200 0.028299700003117323 0.002894944956060499 0.004434845605865121
250 0.04036820000328589 0.002452093905885704 0.003574142404831946
300 0.020175199999357574 0.002292749383486807 0.003249173518270254
350 0.0285645999974804 0.0022251859068637713 0.003105103694833815
400 0.02693639999779407 0.002186765148071572 0.00303598144557327
450 0.019775699998717755 0.0021691000543069094 0.0030005556344985963
500 0.0279649999938556 0.002161238093394786 0.0029818030772730706
550 0.02476060000481084 0.002154053211561404 0.0029716020543128253
600 0.031028500001411885 0.0021524752979166805 0.002966330898925662
650 0.02786329999798909 0.0021515923115657644 0.0029634798085317014
700 0.027741300000343472 0.00214944762585219 0.0029621868999674915
750 0.026984

In [305]:
dataFile = "testdata"  
data = scio.loadmat(dataFile)
inpArr = data['testdata']
x = np.array(inpArr, dtype=np.float32)
x = x.reshape(x.shape[1],1)
x = torch.from_numpy(x).cuda()
out=model(x)
out=out.cpu()
scio.savemat('sxtest.mat',{'sxtest':out.detach().numpy()})